# Reproducing Brezovec et al. (2022) from dandiset 000727

**Paper.** Brezovec LE, Berger AB, Druckmann S, Clandinin TR (2022). Mapping the neural dynamics of
locomotion across the *Drosophila* brain. bioRxiv.
[doi:10.1101/2022.03.20.485047](https://doi.org/10.1101/2022.03.20.485047). This attempt used the
full text of the preprint, version 1 (21 March 2022), which is open access (CC BY-NC-ND) and the
only version on bioRxiv. The paper later appeared in *Current Biology* (doi:10.1016/j.cub.2023.12.063),
which is not open access and was not used.

**Dandiset.** [DANDI:000727](https://dandiarchive.org/dandiset/000727/0.240106.0043), version
0.240106.0043. The paper cites no DANDI version (its data statement points to Dryad and gives the
DOIs in a key resources table that is not in the preprint). This is the only published version.
The draft holds the same 18 assets (same blobs and sizes); it differs only in metadata added later
(keywords, contributor details, and links to the conversion code and an example notebook), none
of which the analysis uses.

**What the dandiset holds.** One file per fly for nine flies: raw GCaMP6f and tdTomato volumes
(256 x 128 x 49 voxels, 3384 volumes), the GCaMP6f volumes after motion correction, high-pass
filtering and z-scoring, warped into the Functional Drosophila Atlas (FDA; 314 x 146 x 91 voxels),
an anatomical scan, and the FicTrac output of the ball at 50 Hz. It does not hold the paper's
mean brain, a neuropil atlas for any of these spaces, the supervoxels, the per-plane acquisition
times, or the motion-corrected tdTomato channel.

**How the data are used.** The analysis runs on the processed (FDA) volumes, since motion
correction and registration with ANTs cannot be repeated here. Twelve of the 91 FDA planes, every
seventh from plane 10, are streamed for all nine flies (22 GB compressed); the files total
610 GB.
The paper's supervoxels are replaced by 6 x 6 voxel blocks within each plane. Behaviour follows the
paper's code (`ClandininLab/brezovec_volition_2023`, commit 8e2b275). No data from outside the
dandiset are used.

| Claim | Tested |
|---|---|
| dataset-structure: nine flies, 30,456 volumes at 1.8 Hz, 1.6M voxels, 50 Hz tracking | yes |
| fig2c-fraction-correlated: 39% of the brain correlates with velocity | yes |
| fig2c-venn-categories: 15% forward only, 20% forward and turning, 4% turning only | yes |
| fig2-symmetry: forward symmetric, turning anti-symmetric | yes |
| fig2e-region-fractions, fig3-topographic-maps | blocked: no neuropil labels |
| fig4bcd-pc-structure: PC1 state, PC2 forward, PC3 turn sign | yes |
| fig4fg-ridge-r2: R2 about 0.3 with 100 PCs | yes |
| fig4h-r2-vs-movement: flies that move more are better predicted | yes |
| fig4i-weights-stereotyped | yes |
| fig5a-unique-variance: forward uniquely encoded, walking state not | yes |
| fig5c-global-below-threshold | yes |
| fig5f-ipsi-contra: ipsilateral increase, contralateral decrease | yes |
| fig6-peak-time-range: peaks from 300 ms before to over 1 s after | yes |
| fig6h-peak-vs-width | yes |
| fig6fg-ipsi-leads-contra | yes |
| fig6eg-region-sequence, fig7-within-region-gradients | blocked: no neuropil labels |
| figs2-tdtomato-control | blocked: no motion-corrected tdTomato |
| figs4a-supervoxel-number | blocked: too large |

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from scipy import signal, stats

import brainwide as bw

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = bw.load_sessions()
flies = [s["subject_id"] for s in sessions]
by_fly = dict(zip(flies, sessions))
times = {f: bw.neural_times(s) for f, s in by_fly.items()}
beh = {f: bw.behaviour_at(s, times[f]) for f, s in by_fly.items()}
beh_start = {f: bw.behaviour_at(s, times[f] - 0.5 / s["processed"]["rate"]) for f, s in by_fly.items()}
print(f"metadata, FicTrac and chunk index of {len(sessions)} files in {time.time() - started:.0f} s")
planes = {}
with ThreadPoolExecutor(bw.N_THREADS) as pool:
    for f, s in by_fly.items():
        regressors = np.array([beh[f]["fwd"], beh[f]["left"], beh[f]["right"],
                               beh_start[f]["fwd"], beh_start[f]["left"], beh_start[f]["right"]])
        planes[f] = bw.load_fly(s, regressors, pool)
print(f"streamed {len(bw.SLICES)} FDA planes of {len(flies)} flies from dandiset {bw.DANDISET_ID} "
      f"version {bw.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
with DandiAPIClient() as client:
    meta = client.get_dandiset(bw.DANDISET_ID, bw.DANDISET_VERSION).get_raw_metadata()
    sizes = {a.path: a.size for a in client.get_dandiset(bw.DANDISET_ID, bw.DANDISET_VERSION).get_assets()}
summary = meta["assetsSummary"]
print("Archive summary: subjects", summary["numberOfSubjects"], "; files", summary["numberOfFiles"],
      "; species", [s["name"] for s in summary["species"]])
info = pd.DataFrame([{
    "fly": f, "folder": s["path"].split("/")[0], "session_id": s["session_id"], "start": s["session_start"][:19],
    "sex": s["subject"]["sex"], "age": s["subject"]["age"],
    "volumes": s["processed"]["shape"][0], "raw_shape": s["raw"]["shape"][1:], "fda_shape": s["processed"]["shape"][1:],
    "volume_hz": round(s["processed"]["rate"], 5), "raw_hz": round(s["raw"]["rate"], 5),
    "fictrac_frames": s["fictrac"]["SpatialSeriesRotationDeltaLabFrame"]["data"].shape[0],
    "fictrac_hz": s["fictrac"]["SpatialSeriesRotationDeltaLabFrame"]["rate"],
    "anat_start_s": round(s["anatomical"]["start"], 1)} for f, s in by_fly.items()])
print("\nGenotypes:", sorted({s["subject"]["genotype"] for s in sessions}))
print("Species:", sorted({s["subject"]["species"] for s in sessions}))
print("\n", info.to_string(index=False))
print("\nFlies:", len(flies), "; distinct subject ids:", info.fly.nunique(), "; sessions per fly:", info.groupby("fly").size().max())
print("Volumes over all flies:", int(info.volumes.sum()))
nwb_gb = [sizes[s["path"]] / 1e9 for s in sessions]
streamed_gb = [sum(c[2] for z in s["chunk_index"] for c in s["chunk_index"][z]) / 1e9 for s in sessions]
print(f"NWB file sizes {min(nwb_gb):.1f} to {max(nwb_gb):.1f} GB; compressed bytes of the {len(bw.SLICES)} processed planes streamed per fly "
      f"{min(streamed_gb):.2f} to {max(streamed_gb):.2f} GB, {sum(streamed_gb):.1f} GB in all; all files {sum(sizes.values()) / 1e9:.0f} GB")
dur_img = info.volumes / info.volume_hz
dur_fic = info.fictrac_frames / info.fictrac_hz
print(f"Imaging length {dur_img.min():.1f} to {dur_img.max():.1f} s; FicTrac length {dur_fic.min():.1f} to {dur_fic.max():.1f} s")
print("Start times of imaging and FicTrac series (s):",
      sorted({s["processed"]["start"] for s in sessions} | {s["raw"]["start"] for s in sessions}),
      sorted({v["start"] for s in sessions for v in s["fictrac"].values()}))

The archive's summary (9 subjects, 18 files, *Drosophila melanogaster*) matches the files: nine
females of the stated genotype (nSyb-Gal4 driving GCaMP6f and myr::tdTomato), one session each,
with the subject ids `fly_087` to `fly_105` that the paper's code lists (the code also drops a
`fly_095` that failed its quality control; it is not in the dandiset). Each holds 3384 volumes at
1.88 Hz (1800.3 s) and 90,000 FicTrac frames at 50 Hz (1800.0 s), so the 30 minute sessions and
the 30,456 volumes of the paper are there. Imaging and FicTrac both start at 0 s; the conversion
code (`catalystneuro/clandinin-lab-to-nwb`, commit f083b09) wrote FicTrac with uniform
timestamps from 0 because synchronisation data were missing, and the paper's code makes the same
assumption (FicTrac frame 0 at the start of imaging). The clock check below tests it. The
anatomical scan starts 1884 to 2240 s after the functional scan, so it was taken after it, as
the Methods say. The age field (P3D/P4D) gives the 3 to 4 days after eclosion of the Methods.

In [ ]:
s0 = sessions[0]
print("Raw functional:", s0["raw"]["shape"], s0["raw"]["dtype"], "grid spacing", s0["raw"]["grid_spacing"], s0["raw"]["grid_spacing_unit"])
print("  voxels per volume:", int(np.prod(s0["raw"]["shape"][1:])))
print("Processed:", s0["processed"]["shape"], s0["processed"]["dtype"], "chunks", s0["processed"]["chunks"],
      "grid spacing", s0["processed"]["grid_spacing"], repr(s0["processed"]["grid_spacing_unit"]))
print("  description:", s0["processed"]["description"])
print("Every file has the same shapes:", len({(s["raw"]["shape"], s["processed"]["shape"]) for s in sessions}) == 1)
print("\nFicTrac series (unit, conversion, description):")
for name, v in s0["fictrac"].items():
    print(f"  {name:40s} {str(v['data'].shape):12s} {v['unit']:7s} {v['conversion']:<7} {v['description'][:110]}")
rows = []
for f, s in by_fly.items():
    ft = s["fictrac"]
    d = ft["SpatialSeriesRotationDeltaLabFrame"]["data"]
    speed = ft["SpatialSeriesMovementSpeed"]["data"][:, 0]
    err = ft["SpatialSeriesRotationDeltaError"]["data"][:, 0]
    v = bw.smoothed_velocity(s)
    rows.append({"fly": f, "dRotLab_x_1_99": np.round(np.percentile(d[:, 0], [1, 99]), 3).tolist(),
                 "dRotLab_y_1_99": np.round(np.percentile(d[:, 1], [1, 99]), 3).tolist(),
                 "dRotLab_z_1_99": np.round(np.percentile(d[:, 2], [1, 99]), 3).tolist(),
                 "speed_vs_xy_norm_r": round(np.corrcoef(speed, np.hypot(d[:, 0], d[:, 1]))[0, 1], 5),
                 "delta_error_median": round(float(np.median(err)), 0),
                 "fwd_mm_s_1_99": np.round(np.percentile(v["forward_mm_s"], [1, 99]), 1).tolist(),
                 "rot_deg_s_1_99": np.round(np.percentile(v["rotation_deg_s"], [1, 99]), 0).tolist(),
                 "walking_pct": round(100 * bw.walking_state(v["Y"], v["Z"]).mean(), 1),
                 "mean_VF_mm_s": round(float(np.clip(v["forward_mm_s"], 0, None).mean()), 2)})
beh_table = pd.DataFrame(rows)
print("\n", beh_table.to_string(index=False))

The raw volumes are 256 x 128 x 49 voxels of 2.61 x 2.61 x 5 um, as the Methods state (1,605,632
voxels per volume). The processed series is 314 x 146 x 91 voxels with a grid spacing of 1, 1, 1 in
unit "unknown", so its voxel size is not given; its description names the FDA, a space the
preprint does not use (the preprint's analyses are in its own mean brain, of the raw volume's
256 x 128 x 49 shape).

Every FicTrac series is stored with unit "meters" and conversion 0.0045 (the ball radius), while
the descriptions give radians for angles and positions and "radians per frame" for speed. The
values are radians (per frame for the deltas): the 1st to 99th percentiles of the lab-frame
deltas lie within 0.14 rad, and the movement speed equals the norm of the roll and pitch deltas
(r = 1.0). Multiplying a
heading or an orientation by the conversion gives no meaningful length, so the unit and
conversion are wrong for those series. `SpatialSeriesRotationDeltaError`, described as an error in
radians, has medians of 6067 to 10,435, which cannot be radians. With the 4.5 mm radius and
50 Hz, the 99th percentile of forward velocity is 2.5 to 14.8 mm/s and rotation reaches 92 to
371 deg/s (1st and 99th percentiles above), plausible for a tethered fly; fly_097 walks least
(mean VF 0.21 mm/s).

In [ ]:
import hashlib
digest = {f: hashlib.md5(s["fictrac"]["SpatialSeriesRotationDeltaLabFrame"]["data"].tobytes()).hexdigest()[:12]
          for f, s in by_fly.items()}
print("Distinct FicTrac series:", len(set(digest.values())), "of", len(digest))
fwd = np.array([bw.smoothed_velocity(s)["forward_mm_s"] for s in sessions])
cc = np.corrcoef(fwd)
print("Largest correlation of forward velocity between two flies:", round(float(np.max(cc[np.triu_indices(9, 1)])), 3))
img = {f: hashlib.md5(s["h5py_check"].tobytes()).hexdigest()[:12] for f, s in by_fly.items()}
print("Distinct processed blocks (first 54 volumes of plane", bw.SLICES[len(bw.SLICES) // 2], "):", len(set(img.values())), "of", len(img))
check_z = bw.SLICES[len(bw.SLICES) // 2]
print("Direct chunk reads equal the h5py values in every file:",
      [planes[f][check_z].get("same_as_h5py") for f in flies])

No recording is deposited twice: the nine FicTrac series and the nine processed volumes differ,
and no two flies' forward velocities are correlated. The direct reads of compressed chunks used
here give the same values as h5py.

In [ ]:
Z = bw.SLICES
n_vol = {f: by_fly[f]["processed"]["shape"][0] for f in flies}
N = sum(n_vol.values())
inside = np.array([[planes[f][z]["inside"] for z in Z] for f in flies])   # fly, plane, x, y
brain = inside.all(0)
print("FDA planes used:", list(Z))
print("In-brain voxels per plane (in all 9 flies):", dict(zip(Z, brain.sum((1, 2)).tolist())))
print(f"In the brain of all 9 flies: {brain.sum()} voxels; of at least one: {inside.any(0).sum()}; "
      f"per fly {inside.sum((1, 2, 3)).min()} to {inside.sum((1, 2, 3)).max()}")
rows = []
for i, f in enumerate(flies):
    sx = np.array([planes[f][z]["sx"] for z in Z])[inside[i]]
    sxx = np.array([planes[f][z]["sxx"] for z in Z])[inside[i]]
    mu = sx / n_vol[f]
    sd = np.sqrt(sxx / n_vol[f] - mu ** 2)
    rows.append({"fly": f, "mean_abs_max": float(np.abs(mu).max()), "sd_1": np.percentile(sd, 1),
                 "sd_median": np.median(sd), "sd_99": np.percentile(sd, 99), "sd_max": sd.max()})
print("\nPer-voxel mean and s.d. over time of the processed series, in-brain voxels:")
print(pd.DataFrame(rows).round(4).to_string(index=False))
# Left-right axis: mirror symmetry of the brain mask on each in-plane axis
both_any = inside.any(0)
for axis, size in ((1, 314), (2, 146)):
    best = []
    for c2 in range(size - 12, size + 13):     # mirror x -> c2 - x, c2 = twice the centre
        idx = c2 - np.arange(size)
        ok = (idx >= 0) & (idx < size)
        a = np.take(both_any, np.arange(size)[ok], axis=axis)
        b = np.take(both_any, idx[ok], axis=axis)
        best.append((float((a & b).sum() / max((a | b).sum(), 1)), c2 / 2))
    best.sort(reverse=True)
    print(f"Axis of {size} voxels: best mirror centre {best[0][1]} (overlap {best[0][0]:.3f}); centre {(size - 1) / 2} gives "
          f"{[o for o, c in best if c == (size - 1) / 2][0]:.3f}")

The processed values are consistent with the description: every in-brain voxel has a mean of 0
over time (to the four decimals printed) and a s.d. below 1 (at most 0.992), as expected if each voxel was z-scored and then warped
(interpolation lowers the spread; the Methods z-score before alignment). Voxels outside each
fly's brain are constant 0. The in-brain masks differ between flies; the analysis uses voxels in
the brain of all nine. On the 314-voxel axis the mask is close to mirror-symmetric about its
centre, 156.5 (overlap 0.914), and less so on the 146-voxel axis (best 0.665), so the 314-voxel
axis is left-right and its centre is the midline; the maps and blocks are mirrored about 156.5. Which
half is the fly's left is not stated anywhere in the dandiset.

In [ ]:
# Clock check. Brain-wide mean activity against walking speed, and the left-minus-right hemisphere
# difference against signed rotation, with the behaviour shifted in time. A shared clock gives a
# peak at a small positive lag (GCaMP6f follows behaviour); a broken one gives no peak near 0.
rng = np.random.default_rng(0)
nbx, nby = bw.block_layout()
counts = np.array([[planes[f][z]["counts"] for z in Z] for f in flies])      # fly, plane, block
keep = (counts >= bw.BLOCK ** 2 / 2).all(0)                                    # plane, block
bx_of = np.tile(np.arange(nbx * nby) // nby, (len(Z), 1))
lags = np.arange(-10, 10.01, 0.2)
rows = []
for f, s in by_fly.items():
    blocks = np.concatenate([planes[f][z]["blocks"][:, keep[i]] for i, z in enumerate(Z)], 1)
    bxs = bx_of[keep]
    glob = blocks.mean(1)
    lr = blocks[:, bxs < nbx // 2].mean(1) - blocks[:, bxs >= nbx // 2].mean(1)
    v = bw.smoothed_velocity(s)
    speed = np.abs(v["forward_mm_s"])
    out = {"fly": f}
    for name, sig, src in (("global~speed", glob, speed), ("halfdiff~rotation", lr, v["rotation_deg_s"])):
        rs = np.array([np.corrcoef(sig, np.interp(times[f] - lag, v["t"], src))[0, 1] for lag in lags])
        k = np.argmax(np.abs(rs))
        null = []
        for _ in range(100):
            shift = rng.uniform(60, 1740)
            null.append(abs(np.corrcoef(sig, np.interp((times[f] - shift) % 1800, v["t"], src))[0, 1]))
        out[name + "_peak_lag_s"] = round(lags[k], 1)
        out[name + "_peak_r"] = round(rs[k], 3)
        out[name + "_null_99pct"] = round(np.percentile(null, 99), 3)
    rows.append(out)
clock = pd.DataFrame(rows)
print("Positive lag: neural activity follows behaviour.")
print(clock.to_string(index=False))

In every fly the brain-wide mean correlates with walking speed most strongly at a lag of 0.4 to
0.8 s (neural activity after behaviour), at r = 0.412 to 0.684, against at most 0.075 for the
99th percentile with the behaviour shifted by a minute or more. In 8 of 9 flies the difference
between the two image halves correlates positively with signed rotation (0.159 to 0.481, at lags
of -0.2 to 0.4 s); in fly_105 the largest correlation is -0.100, at 0.8 s. All exceed their
shifted-behaviour 99th percentiles (at most 0.058). The behaviour and imaging streams are
consistent with one clock, to within the lag of GCaMP6f. A shared start at 0 s is an assumption of
the conversion and of the paper's code; this check supports it to within a few hundred
milliseconds per fly, the resolution set by the volume rate, and cannot test offsets smaller than
that.

## dataset-structure

> Nine flies were imaged for 30 minutes each at a volume rate of about 1.8 Hz, with 256 x 128 x 49
> voxels of 2.6 x 2.6 x 5 um (about 1.6 million voxels per volume), giving 30,456 volumes in all,
> while the ball was tracked at 50 Hz.

In [ ]:
structure = pd.DataFrame([
    ("flies", 9, len(flies), 0),
    ("volumes over all flies", 30456, int(info.volumes.sum()), 0),
    ("volume rate (Hz)", 1.8, round(float(info.volume_hz.mean()), 4), 0.18),
    ("voxels per volume", 1600000, int(np.prod(s0["raw"]["shape"][1:])), 160000),
    ("ball tracking rate (Hz)", 50, float(info.fictrac_hz.mean()), 0),
], columns=["quantity", "expected", "measured", "tolerance"])
structure["within"] = (structure.measured - structure.expected).abs() <= structure.tolerance
print(structure.to_string(index=False))
print("Voxel size (um):", np.round(np.array(s0["raw"]["grid_spacing"]) * 1e6, 2).tolist(),
      "; session length (min):", round(float(dur_img.mean()) / 60, 2))

**Result: reproduced.** The dandiset holds 9 flies with 30,456 volumes in all, at 1.8797 Hz, of
1,605,632 voxels of 2.61 x 2.61 x 5 um, over 30.0 minutes, with ball tracking at 50 Hz; every value
is within its tolerance.

## Voxel correlations with velocity (Figure 2)

The correlation of each voxel with VF, VL and VR, pooled over the nine flies' 30,456 volumes, is
computed from per-fly sums kept while streaming. The thresholds of the paper are |r| for
two-sided p < 1e-4 (single voxels) and p < 1e-8 (supervoxels, Bonferroni over 2000 x 49). As in
the paper, the p-values treat time points as independent.

In [ ]:
def pooled_maps(k_offset=0):
    sx = sum(np.array([planes[f][z]["sx"] for z in Z]) for f in flies)
    sxx = sum(np.array([planes[f][z]["sxx"] for z in Z]) for f in flies)
    sxb = sum(np.array([planes[f][z]["sxb"] for z in Z]) for f in flies)       # plane, k, x, y
    src = beh if k_offset == 0 else beh_start
    out = []
    for k, key in enumerate(("fwd", "left", "right")):
        b = np.concatenate([src[f][key] for f in flies])
        out.append(bw.pooled_r(sx, sxx, sxb[:, k + k_offset], b.sum(), (b ** 2).sum(), N))
    return np.nan_to_num(np.array(out))                                          # k, plane, x, y

r_vox = pooled_maps(0)
r_vox_start = pooled_maps(3)
thr_vox = bw.r_threshold(1e-4, N)
thr_sv = bw.r_threshold(1e-8, N)
print(f"N = {N}; |r| threshold for p < 1e-4: {thr_vox:.4f}; for p < 1e-8: {thr_sv:.4f}")
for k, key in enumerate(("VF", "VL", "VR")):
    vals = r_vox[k][brain]
    print(f"{key}: in-brain voxel r from {vals.min():.3f} to {vals.max():.3f}, median {np.median(vals):.3f}; "
          f"{(vals > thr_vox).mean() * 100:.1f}% above +threshold, {(vals < -thr_vox).mean() * 100:.2f}% below -threshold")

# Block signals pooled over flies (the supervoxels of this attempt)
X = np.concatenate([np.concatenate([planes[f][z]["blocks"][:, keep[i]] for i, z in enumerate(Z)], 1) for f in flies], 0)
X -= X.mean(0)
block_plane = np.concatenate([np.full(keep[i].sum(), z) for i, z in enumerate(Z)])
block_index = np.concatenate([np.flatnonzero(keep[i]) for i in range(len(Z))])
block_bx = block_index // nby
brain_blocks = np.concatenate([brain[i, bw.X_CROP[0]:bw.X_CROP[1], bw.Y_CROP[0]:bw.Y_CROP[1]]
                               .reshape(nbx, bw.BLOCK, nby, bw.BLOCK).sum((1, 3)).ravel()[keep[i]] for i in range(len(Z))])
V = {key: np.concatenate([beh[f][key] for f in flies]) for key in ("fwd", "left", "right", "walk", "Y", "Z")}
def corr_cols(M, b):
    bc = (b - b.mean()) / b.std()
    return (M * bc[:, None]).mean(0) / M.std(0)
r_blk = np.array([corr_cols(X, V[key]) for key in ("fwd", "left", "right")])
print(f"\nBlocks kept: {X.shape[1]} of {keep.size} (at least half their voxels in the brain of every fly); "
      f"they hold {brain_blocks.sum()} of {brain.sum()} in-brain voxels")
print("Block r ranges:", {key: (round(float(r_blk[k].min()), 3), round(float(r_blk[k].max()), 3)) for k, key in enumerate(("VF", "VL", "VR"))})

## fig2c-fraction-correlated

> About 39% of the brain volume correlates with at least one of forward velocity, left and right
> angular velocity.

In [ ]:
def categories(sig, weights=None):
    F, L, R = sig
    w = np.ones(F.shape) if weights is None else weights
    any_ = F | L | R
    tot = w.sum()
    return {"correlated %": 100 * w[any_].sum() / tot,
            "VF only %": 100 * w[F & ~L & ~R].sum() / tot,
            "VF and turning %": 100 * w[F & (L | R)].sum() / tot,
            "turning only %": 100 * w[~F & (L | R)].sum() / tot,
            "all three, % of correlated": 100 * w[F & L & R].sum() / max(w[any_].sum(), 1)}

readings = {
    "(a) voxels, p<1e-4": categories(np.abs(r_vox[:, brain]) > thr_vox),
    "(b) blocks, p<1e-8": categories(np.abs(r_blk) > thr_sv, brain_blocks),
    "variant: voxels, behaviour at volume start": categories(np.abs(r_vox_start[:, brain]) > thr_vox),
    "variant: blocks, p<1e-8, volume start": categories(np.abs(np.array([corr_cols(X, np.concatenate([beh_start[f][k] for f in flies]))
                                                                for k in ("fwd", "left", "right")])) > thr_sv, brain_blocks),
    "variant: voxels in the brain of any fly": categories(np.abs(r_vox[:, inside.any(0)]) > thr_vox),
    "variant: voxels, positive r only": categories(r_vox[:, brain] > thr_vox),
    "variant: blocks, p<1e-4": categories(np.abs(r_blk) > thr_vox, brain_blocks),
}
cat_table = pd.DataFrame(readings).T.round(1)
print(cat_table.to_string())
print("\nExpected: correlated 39 (35.1 to 42.9)")
per_plane = pd.DataFrame({"plane": Z, "in-brain voxels": brain.sum((1, 2)),
                          "correlated % (a)": [100 * (np.abs(r_vox[:, i][:, brain[i]]) > thr_vox).any(0).mean() for i in range(len(Z))]})
print("\nBy plane, reading (a):")
print(per_plane.round(1).to_string(index=False))
# Per fly: correlations within each fly alone (no pooling), fraction above the threshold for one fly's n
thr_fly = bw.r_threshold(1e-4, 3384)
rows = []
for f in flies:
    sx = np.array([planes[f][z]["sx"] for z in Z]); sxx = np.array([planes[f][z]["sxx"] for z in Z])
    sxb = np.array([planes[f][z]["sxb"] for z in Z])
    rs = [bw.pooled_r(sx, sxx, sxb[:, k], beh[f][key].sum(), (beh[f][key] ** 2).sum(), n_vol[f])
          for k, key in enumerate(("fwd", "left", "right"))]
    rs = np.nan_to_num(np.array(rs))[:, brain]
    rows.append({"fly": f, "correlated % (p<1e-4 within fly)": 100 * (np.abs(rs) > thr_fly).any(0).mean(),
                 "VF %": 100 * (np.abs(rs[0]) > thr_fly).mean(), "mean VF (mm/s)": beh_table.set_index("fly").loc[f, "mean_VF_mm_s"]})
print("\nPer fly (each fly's own correlations, threshold for 3384 points):")
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: partially reproduced.** Under reading (a), 31.1% of in-brain voxels correlate with
VF, VL or VR at p < 1e-4, below the expected 35.1 to 42.9% and above half of 39%; under reading
(b), the blocks significant at p < 1e-8 cover 54.9%, above it. No reading is within tolerance.
With behaviour sampled at the start of each volume in place of its midpoint, reading (a) gives
39.2%, so the value depends on neural times that the dandiset does not hold. The correlated
fraction is 0.0 to 7.5% in the outermost planes (10, 80, 87) and 21.0 to 39.1% in the others.
Within single flies, with the threshold for 3384 points, 2.4 to 19.7% of voxels pass.

## fig2c-venn-categories

> Of the brain volume, about 15% correlates only with forward velocity, 20% with forward velocity
> and turning, and 4% only with turning; only about 10% of correlated voxels correlate with all
> three components, so 90% show a tuning preference.

In [ ]:
expected_venn = {"VF only %": (15, 1.5), "VF and turning %": (20, 2.0), "turning only %": (4, 0.4),
                 "all three, % of correlated": (10, 1.0)}
rows = []
for name, vals in readings.items():
    row = {"reading": name}
    for q, (e, tol) in expected_venn.items():
        row[q] = round(vals[q], 1)
        row[q + " ok"] = abs(vals[q] - e) <= tol
    row["ranking holds"] = vals["VF and turning %"] > vals["VF only %"] > vals["turning only %"]
    row["all three < 20% of correlated"] = vals["all three, % of correlated"] < 20
    rows.append(row)
venn = pd.DataFrame(rows)
print(venn.to_string(index=False))
print("\nExpected:", expected_venn)

**Result: partially reproduced.** Under reading (a), 11.9% of in-brain voxels correlate with VF
only (expected 15), 15.9% with VF and turning (20), 3.2% with turning only (4), and 11.5% of the
correlated voxels with all three (10). None is within its tolerance, but the ranking holds
(forward and turning above forward only, above turning only) and fewer than 20% of correlated
voxels correlate with all three. Under reading (b) forward and turning is 40.4% and all three
32.2% of correlated volume. With behaviour at the start of each volume, reading (a) gives 11.6,
24.1, 3.5 and 25.7%.

## fig2-symmetry

> Forward velocity correlations are mirror-symmetric across the midline, while left and right
> turning correlations are anti-symmetric, with more correlated voxels in the hemisphere
> ipsilateral to the turn.

In [ ]:
xs = np.arange(314)
xm = 313 - xs                        # mirror about 156.5
pair = brain & brain[:, xm, :]
def mirror(m):
    return m[:, xm, :]
sym = {}
for k, key in enumerate(("VF", "VL", "VR")):
    sym[key + " vs own mirror"] = np.corrcoef(r_vox[k][pair], mirror(r_vox[k])[pair])[0, 1]
sym["VL vs mirrored VR"] = np.corrcoef(r_vox[1][pair], mirror(r_vox[2])[pair])[0, 1]
print("Correlation of voxel r maps over in-brain mirror pairs:", {k: round(v, 3) for k, v in sym.items()})
halfA = (xs < 157)[None, :, None] & brain
halfB = (xs >= 157)[None, :, None] & brain
sig_counts = pd.DataFrame({key: {"half A (x < 157)": int(((np.abs(r_vox[k]) > thr_vox) & halfA).sum()),
                                 "half B (x >= 157)": int(((np.abs(r_vox[k]) > thr_vox) & halfB).sum())}
                           for k, key in enumerate(("VF", "VL", "VR"))}).T
sig_counts["A / B"] = sig_counts.iloc[:, 0] / sig_counts.iloc[:, 1]
print("\nIn-brain voxels with p < 1e-4, by image half (in-brain voxels: A", int(halfA.sum()), ", B", int(halfB.sum()), "):")
print(sig_counts.round(2).to_string())
ipsi_half = {"left": "A" if sig_counts.loc["VL", "A / B"] > 1 else "B", "right": "A" if sig_counts.loc["VR", "A / B"] > 1 else "B"}
crit = {"(i) VF mirror r >= 0.7": sym["VF vs own mirror"] >= 0.7,
        "(ii) VL~mirrored VR > VL~own mirror": sym["VL vs mirrored VR"] > sym["VL vs own mirror"],
        "(iii) VL and VR >= 2x in opposite halves": (max(sig_counts.loc["VL", "A / B"], 1 / sig_counts.loc["VL", "A / B"]) >= 2
                                                     and max(sig_counts.loc["VR", "A / B"], 1 / sig_counts.loc["VR", "A / B"]) >= 2
                                                     and ipsi_half["left"] != ipsi_half["right"])}
print("\nCriteria:", crit)
print("Half with more VL voxels:", ipsi_half["left"], "; with more VR voxels:", ipsi_half["right"])
rows = []
for f in flies:
    sx = np.array([planes[f][z]["sx"] for z in Z]); sxx = np.array([planes[f][z]["sxx"] for z in Z])
    sxb = np.array([planes[f][z]["sxb"] for z in Z])
    rs = np.nan_to_num(np.array([bw.pooled_r(sx, sxx, sxb[:, k], beh[f][key].sum(), (beh[f][key] ** 2).sum(), n_vol[f])
                                 for k, key in enumerate(("fwd", "left", "right"))]))
    rows.append({"fly": f, "VF mirror r": np.corrcoef(rs[0][pair], mirror(rs[0])[pair])[0, 1],
                 "VL: sum r, A - B": rs[1][halfA].sum() - rs[1][halfB].sum(),
                 "VR: sum r, A - B": rs[2][halfA].sum() - rs[2][halfB].sum()})
print("\nPer fly:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
zi = list(Z).index(59)
for k, (ax, key) in enumerate(zip(axes, ("VF", "VL", "VR"))):
    im = ax.imshow(np.where(brain[zi], r_vox[k][zi], np.nan).T, cmap="RdBu_r", vmin=-0.3, vmax=0.3, origin="lower")
    ax.set_title(f"{key}, FDA plane 59"); ax.set_xlabel("x (left-right, voxels)")
fig.colorbar(im, ax=axes, label="pooled r")
plt.show()
vals = r_vox[:, zi][:, brain[zi]]
print(f"Plane 59 r range {vals.min():.3f} to {vals.max():.3f}; voxels outside the colour range +-0.3: {(np.abs(vals) > 0.3).sum()}")

**Result: reproduced.** The VF correlation map correlates with its mirror image at r = 0.823. The
VL map correlates with the mirrored VR map at 0.672, more than with its own mirror image
(0.449). Voxels significant for VL are 2.38 times as many in half A as in half B, and voxels
significant for VR 0.31 times as many (3.2 times as many in half B). In single flies the VF map
correlates with its mirror at 0.166 to 0.580; summed VL correlation is larger in half A in 8 of 9
flies and summed VR correlation larger in half B in 7 of 9. If turning correlates with the
ipsilateral hemisphere, as the paper says, half A (x < 157) is the fly's left; the dandiset does
not state which half is left, so the ipsilateral part is not tested.

## Principal components of the pooled blocks (Figure 4)

PCA of the block signals of all nine flies concatenated (the "superfly"), by eigendecomposition
of the covariance matrix, as pca.py does on the paper's supervoxels.

In [ ]:
t0 = time.time()
cov = (X.T @ X) / (X.shape[0] - 1)
evals, evecs = np.linalg.eigh(cov.astype(np.float64))
evals, evecs = evals[::-1], evecs[:, ::-1]
del cov
N_PCS = 500
pcs = X @ evecs[:, :N_PCS].astype(np.float32)
for i, key in ((0, "fwd"), (1, "fwd"), (2, "left")):     # orientation fixed in the criterion
    if np.corrcoef(pcs[:, i], V[key])[0, 1] < 0:
        pcs[:, i] *= -1
        evecs[:, i] *= -1
print(f"PCA of {X.shape[0]} x {X.shape[1]} in {time.time() - t0:.0f} s; variance explained by PCs 1-10 (%):",
      np.round(100 * evals[:10] / evals.sum(), 2).tolist())

## fig4bcd-pc-structure

> The first principal components of brain-wide activity are structured by behaviour; PC1 is low
> when the fly is stopped and high otherwise, PC2 increases with forward velocity and not with
> angular velocity, and PC3 carries the sign of angular velocity.

In [ ]:
def split_corr(sig, comp, reading):
    """Correlation of sig with comp below and above 0.75 s.d. of comp, with p-values."""
    z = comp / comp.std() if reading == "a" else (comp - comp.mean()) / comp.std()
    out = {}
    for side, m in (("below", z <= 0.75), ("above", z > 0.75)):
        r, p = stats.pearsonr(sig[m], comp[m])
        out[side] = (r, p, int(m.sum()))
    return out

pc_rows, pc_checks = [], {}
for reading in ("a", "b"):
    res = {}
    for i in range(3):
        for key in ("fwd", "left", "right"):
            sc = split_corr(pcs[:, i], V[key], reading)
            res[(i, key)] = sc
            pc_rows.append({"reading": reading, "PC": i + 1, "component": key,
                            "r below": sc["below"][0], "p below": sc["below"][1], "n below": sc["below"][2],
                            "r above": sc["above"][0], "p above": sc["above"][1], "n above": sc["above"][2]})
    c1 = res[(0, "fwd")]["below"][0] > 0 and all(abs(res[(0, "fwd")]["below"][0]) > abs(res[(j, "fwd")]["below"][0]) for j in (1, 2))
    c2 = (np.argmax([res[(j, "fwd")]["above"][0] for j in range(3)]) == 1
          and abs(res[(1, "left")]["above"][0]) < abs(res[(1, "fwd")]["above"][0])
          and abs(res[(1, "right")]["above"][0]) < abs(res[(1, "fwd")]["above"][0]))
    l3, r3 = res[(2, "left")]["above"], res[(2, "right")]["above"]
    c3 = np.sign(l3[0]) != np.sign(r3[0]) and l3[1] < 0.05 and r3[1] < 0.05
    pc_checks[reading] = {"(i) PC1 below-threshold": c1, "(ii) PC2 forward above": c2, "(iii) PC3 turn sign": c3}
print(pd.DataFrame(pc_rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriteria by reading:", pc_checks)
# Which of the first 10 PCs best match each role (all time points)
corr10 = pd.DataFrame({key: [np.corrcoef(pcs[:, i], V[key])[0, 1] for i in range(10)] for key in ("fwd", "left", "right", "walk")},
                      index=[f"PC{i + 1}" for i in range(10)])
corr10["left - right"] = corr10["left"] - corr10["right"]
print("\nCorrelations over all time points, PCs 1-10:")
print(corr10.round(3).to_string())
# PCs by fly: correlation of PC1-3 with the components within each fly
rows = []
for j, f in enumerate(flies):
    sl = slice(j * 3384, (j + 1) * 3384)
    rows.append({"fly": f, **{f"PC{i + 1}~{key}": np.corrcoef(pcs[sl, i], V[key][sl])[0, 1]
                             for i in range(3) for key in ("fwd", "left", "right")}})
print("\nPer fly:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for i, ax in enumerate(axes):
    ax.hexbin(V["Y"] * bw.BALL_RADIUS_MM * bw.FICTRAC_HZ, V["Z"] * 180 / np.pi * bw.FICTRAC_HZ, C=pcs[:, i],
              gridsize=30, extent=(-5, 15, -300, 300), reduce_C_function=np.mean, cmap="viridis")
    ax.set_title(f"mean PC{i + 1} in velocity space"); ax.set_xlabel("forward (mm/s)"); ax.set_ylabel("rotation (deg/s, + left)")
plt.show()
fv, rv = V["Y"] * bw.BALL_RADIUS_MM * bw.FICTRAC_HZ, V["Z"] * 180 / np.pi * bw.FICTRAC_HZ
print("Points outside the axes (forward -5 to 15 mm/s, rotation -300 to 300 deg/s):",
      int(((fv < -5) | (fv > 15) | (np.abs(rv) > 300)).sum()), "of", len(fv))

**Result: reproduced.** All three parts hold under both readings of the threshold. Under reading
(a), PC1's below-threshold correlation with VF is 0.202, against 0.065 for PC2 and -0.045 for
PC3. Above the threshold PC2 has the largest correlation with VF (0.272; PC1 0.162, PC3 0.086),
with 0.091 for VL and 0.087 for VR. PC3's above-threshold correlations are 0.114 with VL and
-0.259 with VR (p = 3.5e-14 and 5.6e-72). PC1 explains 2.22% of the block variance. Within each
fly, over all time points, PC1 correlates positively with VF (0.22 to 0.57) and PC3 more with VL
than with VR, in 9 of 9 flies.

## fig4fg-ridge-r2

> Linear models on the principal components predict forward, left and right velocity on held-out
> data with average R2 of about 0.3 using 100 PCs, performance improving over the first 100 PCs
> before declining.

In [ ]:
ALPHAS = [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6, 1e7]   # pca_fit_behavior_models.py
N_PC_LIST = [1, 10, 30, 100, 250, 500]
folds = bw.contiguous_folds(3384, 5)
t0 = time.time()

def fly_scores(scores_source):
    rows, best_alpha = [], {}
    for j, f in enumerate(flies):
        P_f = scores_source(j)
        Y = np.column_stack([beh[f][key] for key in ("fwd", "left", "right")])
        for n in N_PC_LIST:
            sc_all = bw.ridge_scores(P_f[:, :n], Y, folds, ALPHAS).mean(0)      # alpha, component
            for c, key in enumerate(("fwd", "left", "right")):
                sc = sc_all[:, c]
                rows.append({"fly": f, "component": key, "n_pcs": n, "R2": sc.max(), "alpha": ALPHAS[int(np.argmax(sc))]})
                best_alpha[(f, key, n)] = ALPHAS[int(np.argmax(sc))]
    return pd.DataFrame(rows), best_alpha

ridge, best_alpha = fly_scores(lambda j: pcs[j * 3384:(j + 1) * 3384])
r2_100 = ridge[ridge.n_pcs == 100].pivot(index="fly", columns="component", values="R2")[["fwd", "left", "right"]]
print(f"Fits done in {time.time() - t0:.0f} s; R2 with 100 superfly PCs, per fly:")
print(r2_100.round(3).to_string())
summary_r2 = pd.DataFrame({"(a) median": r2_100.median(), "(b) mean": r2_100.mean()}).T
print("\nAverage over flies (expected 0.3, 0.27 to 0.33):")
print(summary_r2.round(3).to_string())
print("\nMean R2 over flies by number of PCs:")
curve = ridge.groupby(["component", "n_pcs"]).R2.mean().unstack()
print(curve.round(3).to_string())
print("Chosen alphas at 100 PCs:", ridge[ridge.n_pcs == 100].alpha.value_counts().to_dict())

In [ ]:
# Variant 1: alpha chosen within the training folds (inner contiguous 4-fold), not on the test fold.
t0 = time.time()
rows = []
for j, f in enumerate(flies):
    P_f = pcs[j * 3384:(j + 1) * 3384, :100]
    for key in ("fwd", "left", "right"):
        y = beh[f][key]
        sc = []
        for test in folds:
            train = np.setdiff1d(np.arange(3384), test)
            inner = [train[ix] for ix in bw.contiguous_folds(len(train), 4)]
            inner_sc = bw.ridge_scores(P_f[train], y[train], [np.searchsorted(train, ix) for ix in inner], ALPHAS)
            a = ALPHAS[int(np.argmax(inner_sc.mean(0)))]
            sc.append(bw.ridge_scores(P_f, y, [test], [a])[0, 0])
        rows.append({"fly": f, "component": key, "R2": np.mean(sc)})
nested = pd.DataFrame(rows).pivot(index="fly", columns="component", values="R2")[["fwd", "left", "right"]]
print(f"Variant, nested alpha ({time.time() - t0:.0f} s): median {nested.median().round(3).to_dict()}, mean {nested.mean().round(3).to_dict()}")
# Variant 2: PCs of each fly alone (Figure 4G dashed lines)
t0 = time.time()
own = []
for j, f in enumerate(flies):
    Xf = X[j * 3384:(j + 1) * 3384]
    Xf = Xf - Xf.mean(0)
    g_vals, g_vecs = np.linalg.eigh((Xf @ Xf.T).astype(np.float64))     # Gram matrix: same scores as the covariance PCs
    order = np.argsort(g_vals)[::-1][:100]
    own.append((g_vecs[:, order] * np.sqrt(np.clip(g_vals[order], 0, None))).astype(np.float32))
own_r2 = []
for j, f in enumerate(flies):
    sc_all = bw.ridge_scores(own[j], np.column_stack([beh[f][k] for k in ("fwd", "left", "right")]), folds, ALPHAS).mean(0)
    for c, key in enumerate(("fwd", "left", "right")):
        own_r2.append({"fly": f, "component": key, "R2": sc_all[:, c].max()})
own_r2 = pd.DataFrame(own_r2).pivot(index="fly", columns="component", values="R2")[["fwd", "left", "right"]]
print(f"Variant, each fly's own 100 PCs ({time.time() - t0:.0f} s): median {own_r2.median().round(3).to_dict()}, mean {own_r2.mean().round(3).to_dict()}")
# Variant 3: interleaved folds (50 segments of the session, segment i in fold i mod 5)
seg = np.array_split(np.arange(3384), 50)
inter = [np.concatenate(seg[k::5]) for k in range(5)]
rows = []
for j, f in enumerate(flies):
    sc_all = bw.ridge_scores(pcs[j * 3384:(j + 1) * 3384, :100], np.column_stack([beh[f][k] for k in ("fwd", "left", "right")]), inter, ALPHAS).mean(0)
    rows.append({"fly": f, **{key: sc_all[:, c].max() for c, key in enumerate(("fwd", "left", "right"))}})
inter_r2 = pd.DataFrame(rows).set_index("fly")
print(f"Variant, interleaved folds: median {inter_r2.median().round(3).to_dict()}, mean {inter_r2.mean().round(3).to_dict()}")
# Reference: a model with no information (behaviour shifted circularly by half the session) gives
rng_ridge = np.random.default_rng(1)
null = []
for j, f in enumerate(flies):
    for key in ("fwd", "left", "right"):
        y = np.roll(beh[f][key], int(rng_ridge.integers(800, 2584)))
        null.append(bw.ridge_scores(pcs[j * 3384:(j + 1) * 3384, :100], y, folds, ALPHAS).mean(0).max())
print(f"Reference, behaviour rolled by 800 to 2584 volumes: R2 median {np.median(null):.3f}, max {np.max(null):.3f} over 27 fits")

**Result: partially reproduced.** With 100 superfly PCs, the median R2 over flies is 0.155 for VF,
0.196 for VL and 0.212 for VR (reading a; means 0.038, 0.155 and 0.178), all below 0.27 to 0.33
and all at least 0.15. The mean R2 rises up to 100 PCs and declines after it for each component
(VF 0.022, 0.038, 0.036 and 0.033 at 30, 100, 250 and 500 PCs), as the paper describes. Three
flies have negative VF R2 (-0.307, -0.306 and -0.503). With interleaved folds in place of
contiguous ones the medians are 0.318, 0.252 and 0.271; with alpha chosen within the training
folds 0.134, 0.180 and 0.212; with each fly's own PCs 0.151, 0.210 and 0.217. With the behaviour
rolled in time, R2 is at most -0.002.

## fig4h-r2-vs-movement

> Flies that moved less along a velocity component were poorly predicted (R2 near 0), while flies
> that explored it more were well predicted, with R2 up to 0.6.

In [ ]:
amount = pd.DataFrame({f: {"fwd": beh[f]["fwd"].mean() * bw.BALL_RADIUS_MM * bw.FICTRAC_HZ,
                           "left": beh[f]["left"].mean() * 180 / np.pi * bw.FICTRAC_HZ,
                           "right": beh[f]["right"].mean() * 180 / np.pi * bw.FICTRAC_HZ} for f in flies}).T
tab = pd.concat({"mean amount": amount, "R2": r2_100}, axis=1)
print("Per fly: mean VF (mm/s), VL and VR (deg/s), and R2 with 100 PCs")
print(tab.round(3).to_string())
rho = {key: stats.spearmanr(amount[key], r2_100[key]) for key in ("fwd", "left", "right")}
print("\nSpearman per component:", {k: (round(v.statistic, 3), round(v.pvalue, 4)) for k, v in rho.items()})
ra = np.concatenate([stats.rankdata(amount[k]) for k in ("fwd", "left", "right")])
rr = np.concatenate([stats.rankdata(r2_100[k]) for k in ("fwd", "left", "right")])
pooled = stats.pearsonr(ra, rr)
print(f"Within-component ranks pooled over 27 points: r = {pooled.statistic:.3f}, p = {pooled.pvalue:.3g}")
print(f"Largest per-fly R2 {r2_100.values.max():.3f} ({r2_100.stack().idxmax()}); smallest {r2_100.values.min():.3f} ({r2_100.stack().idxmin()})")
print("Expected largest 0.6 (0.54 to 0.66), smallest below 0.1")

**Result: partially reproduced.** The Spearman correlation between a fly's mean amount of a
component and its R2 is positive for all three components (VF 0.017, VL 0.383, VR 0.683; p =
0.042 for VR only), but the pooled correlation of within-component ranks is 0.361 with p =
0.064, short of the p < 0.05 the criterion requires. The largest R2 is 0.534 (fly_094, VF), just
below the band of 0.54 to 0.66, and the smallest is -0.503, below 0.1.

## fig4i-weights-stereotyped

> The model weights given to each PC are stereotyped across flies; PC1 is used for all
> components, PC2 mainly for forward velocity, and PC3 for left and right turning with opposite
> signs.

In [ ]:
W = {}
for j, f in enumerate(flies):
    P_f = pcs[j * 3384:(j + 1) * 3384, :100]
    for key in ("fwd", "left", "right"):
        y = beh[f][key] / beh[f][key].std()
        W[(f, key)] = bw.ridge_fit(P_f, y, best_alpha[(f, key, 100)])[:20]
pair_r = {key: np.median([np.corrcoef(W[(a, key)], W[(b, key)])[0, 1] for i, a in enumerate(flies) for b in flies[i + 1:]])
          for key in ("fwd", "left", "right")}
pc3_opposite = sum(np.sign(W[(f, "left")][2]) != np.sign(W[(f, "right")][2]) for f in flies)
pc2_fwd = sum(abs(W[(f, "fwd")][1]) > max(abs(W[(f, "left")][1]), abs(W[(f, "right")][1])) for f in flies)
pc1_same = sum(len({np.sign(W[(f, k)][0]) for k in ("fwd", "left", "right")}) == 1 for f in flies)
print("Median pairwise r of the 20 weights over 36 fly pairs:", {k: round(v, 3) for k, v in pair_r.items()})
print("PC3 weights of opposite sign for VL and VR:", pc3_opposite, "of 9 flies")
print("PC2 weight largest for VF:", pc2_fwd, "of 9 flies")
print("PC1 weights of one sign for all three:", pc1_same, "of 9 flies")
wt = pd.DataFrame({(f, k): W[(f, k)][:3] for f in flies for k in ("fwd", "left", "right")}, index=["PC1", "PC2", "PC3"]).T
print("\nWeights on PCs 1-3 (behaviour in s.d. units):")
print(wt.round(4).to_string())
fi_checks = {"(i) median pair r >= 0.5 for each": all(v >= 0.5 for v in pair_r.values()),
             "(ii) PC3 opposite in >= 7": pc3_opposite >= 7, "(iii) PC2 forward in >= 7": pc2_fwd >= 7,
             "(iv) PC1 same sign in >= 7": pc1_same >= 7}
print("\nCriteria:", fi_checks)

**Result: reproduced.** The median correlation of the weights on PCs 1 to 20 between pairs of
flies is 0.646 for VF, 0.732 for VL and 0.761 for VR. PC3's weight is positive for VL and
negative for VR in 9 of 9 flies; PC2's weight is largest for VF in 7 of 9; PC1's weights have one
sign for all three components in 9 of 9.

## fig5a-unique-variance

> Demixing with linear models shows that forward velocity is uniquely encoded by a spatially
> organized subset of voxels, while locomotor state (walking or stopped) is not uniquely encoded
> anywhere, although the single-variable maps of forward velocity and locomotor state are very
> similar.

Each behaviour variable is divided by its s.d. before fitting, so that the fixed ridge penalty
(alpha 1) is small next to the 30,456 time points.

In [ ]:
def design(walk_key="walk"):
    cols = [V["fwd"], V["left"], V["right"], V[walk_key]]
    return np.column_stack([c / c.std() for c in cols])

def demix(D, names, alpha=1.0):
    """Mean held-out R2 per block for every model; fold k is the k-th fifth of every fly."""
    fifths = bw.contiguous_folds(3384, 5)
    fold_idx = [np.concatenate([j * 3384 + fifths[k] for j in range(len(flies))]) for k in range(5)]
    models = {"all": list(range(4))}
    models.update({f"without {n}": [i for i in range(4) if i != k] for k, n in enumerate(names)})
    models.update({f"only {n}": [k] for k, n in enumerate(names)})
    r2 = {m: np.zeros(X.shape[1]) for m in models}
    every = np.arange(X.shape[0])
    for test in fold_idx:
        train = np.setdiff1d(every, test)
        dm = D[train].mean(0)
        dc = D[train] - dm
        dtd = dc.T @ dc
        dty = dc.T @ X[train]                    # X is centred; dc sums to 0, so no need to centre X here
        ym = X[train].mean(0)
        yt = X[test]
        sst = ((yt - yt.mean(0)) ** 2).sum(0)
        dt = D[test] - dm
        for m, cols in models.items():
            coef = np.linalg.solve(dtd[np.ix_(cols, cols)] + alpha * np.eye(len(cols)), dty[cols])
            pred = dt[:, cols] @ coef + ym
            r2[m] += 1 - ((yt - pred) ** 2).sum(0) / sst
    return {m: v / len(fold_idx) for m, v in r2.items()}

names = ["VF", "VL", "VR", "walk"]
t0 = time.time()
r2_models = demix(design("walk"), names)
unique = {n: r2_models["all"] - r2_models[f"without {n}"] for n in names}
print(f"Demixing of {X.shape[1]} blocks in {time.time() - t0:.0f} s")
UNIQUE_MIN = 0.01
rows = []
for n in names:
    rows.append({"variable": n, "unique: blocks >= 0.01": int((unique[n] >= UNIQUE_MIN).sum()),
                 "unique: % of blocks": 100 * (unique[n] >= UNIQUE_MIN).mean(),
                 "unique: max": unique[n].max(), "unique: 99th pct": np.percentile(unique[n], 99),
                 "single: median R2": np.median(r2_models[f"only {n}"]), "single: max R2": r2_models[f"only {n}"].max()})
demix_table = pd.DataFrame(rows)
print(demix_table.round(4).to_string(index=False))
print(f"All four: median R2 {np.median(r2_models['all']):.4f}, max {r2_models['all'].max():.4f}; blocks with R2 <= 0: {(r2_models['all'] <= 0).sum()}")
print("Correlation over blocks of single-variable R2, VF against walking state:",
      round(np.corrcoef(r2_models["only VF"], r2_models["only walk"])[0, 1], 3))
for thr in (0.005, 0.01, 0.02):
    print(f"  threshold {thr}: % of blocks encoding uniquely", {n: round(100 * (unique[n] >= thr).mean(), 2) for n in names})
fa = (unique["walk"] >= UNIQUE_MIN).mean(); ff = (unique["VF"] >= UNIQUE_MIN).mean()
print(f"\nCriteria: walking state <= 1%: {fa <= 0.01}; VF >= 5%: {ff >= 0.05}; VF >= 5 x walking: {ff >= 5 * fa}")
# Variant: the walking state as the code intended it (norm of both standardised components)
V["walk_intended"] = np.concatenate([(np.interp(times[f], bw.smoothed_velocity(by_fly[f])["t"],
    bw.walking_state(bw.smoothed_velocity(by_fly[f])["Y"], bw.smoothed_velocity(by_fly[f])["Z"], as_intended=True)) >= 0.5).astype(float)
    for f in flies])
r2_int = demix(design("walk_intended"), names)
u_int = {n: r2_int["all"] - r2_int[f"without {n}"] for n in names}
print("Variant, walking state as intended: % of blocks encoding uniquely",
      {n: round(100 * (u_int[n] >= UNIQUE_MIN).mean(), 2) for n in names})
print("Walking fraction of time points: as run", round(V["walk"].mean(), 3), "; as intended", round(V["walk_intended"].mean(), 3))
# Per fly: the same model within each fly
rows = []
for j, f in enumerate(flies):
    sl = slice(j * 3384, (j + 1) * 3384)
    Df = design("walk")[sl]
    Xf = X[sl] - X[sl].mean(0)
    fifths = bw.contiguous_folds(3384, 5)
    full = bw.multi_ridge_r2(Df, Xf, fifths, 1.0)
    rows.append({"fly": f, **{f"{n} unique %": 100 * ((full - bw.multi_ridge_r2(Df[:, [i for i in range(4) if i != k]], Xf, fifths, 1.0)) >= UNIQUE_MIN).mean()
                              for k, n in enumerate(names)}})
print("\nPer fly (within-fly models):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: partially reproduced.** At the 0.01 threshold, 5 blocks (0.09%) encode walking state
uniquely and 40 (0.75%) encode VF uniquely, 8 times as many, but below the 5% that the criterion
requires. Unique contributions are small everywhere (largest 0.032 for VF and 0.012 for walking
state), and the full model has a median held-out R2 of 0.0019 per block. The single-variable R2
maps of VF and walking state correlate at 0.882 over blocks, the similarity the paper describes.
At a threshold of 0.005, walking state is unique in 1.88% of blocks and VF in 2.24%; with the
walking state the code intended, in 0.97% and 0.91%. Within single flies, walking state is
unique in up to 8.35% of blocks (fly_094).

## fig5c-global-below-threshold

> The functional regions defined by the unique contributions of forward, left and right velocity
> are all correlated with velocity when the fly is below the movement threshold, a widespread
> increase in activity at the initiation of movement, while above the threshold each region
> correlates most with its own velocity component.

In [ ]:
regions = {key: unique[n] >= UNIQUE_MIN for key, n in (("fwd", "VF"), ("left", "VL"), ("right", "VR"))}
print("Blocks per region:", {k: int(v.sum()) for k, v in regions.items()}, "; in two or more regions:",
      int((sum(v.astype(int) for v in regions.values()) > 1).sum()))
reg_sig = {k: X[:, m].mean(1) for k, m in regions.items() if m.any()}
rows, checks5 = [], {}
for reading in ("a", "b"):
    below_ok, above_ok = [], []
    for rk, sig in reg_sig.items():
        best = None
        for key in ("fwd", "left", "right"):
            sc = split_corr(sig, V[key], reading)
            rows.append({"reading": reading, "region": rk, "component": key, "r below": sc["below"][0], "p below": sc["below"][1],
                         "r above": sc["above"][0], "p above": sc["above"][1]})
            below_ok.append(sc["below"][0] > 0 and sc["below"][1] < 0.05)
            if best is None or sc["above"][0] > best[1]:
                best = (key, sc["above"][0])
        above_ok.append(best[0] == rk)
    checks5[reading] = {"all below-threshold r > 0, p < 0.05": all(below_ok) and len(below_ok) == 9,
                        "own component largest above": all(above_ok) and len(above_ok) == 3}
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriteria by reading:", checks5)
rows = []
for j, f in enumerate(flies):
    sl = slice(j * 3384, (j + 1) * 3384)
    row = {"fly": f}
    for rk, sig in reg_sig.items():
        for key in ("fwd", "left", "right"):
            comp = V[key][sl]
            m = comp / V[key].std() <= 0.75
            row[f"{rk}~{key} below"] = np.corrcoef(sig[sl][m], comp[m])[0, 1]
    rows.append(row)
print("\nPer fly, below-threshold r (reading a):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: reproduced.** Under reading (b), all nine below-threshold correlations of the three
regions (40, 4 and 8 blocks) with the three components are positive with p < 0.05 (0.013 to
0.221), and above the threshold each region correlates most with its own component (0.377,
0.170 and 0.266). Under reading (a) one of the nine, the left region with VR, is -0.008 (p =
0.2). The regions are small because few blocks encode a component uniquely
(fig5a-unique-variance).

## fig5f-ipsi-contra

> When the fly turns, neural activity increases in specific regions of the ipsilateral hemisphere
> and decreases in the same regions of the contralateral hemisphere.

In [ ]:
fwd_mm = V["Y"] * bw.BALL_RADIUS_MM * bw.FICTRAC_HZ
rot_deg = V["Z"] * 180 / np.pi * bw.FICTRAC_HZ
slow = (fwd_mm >= 1) & (fwd_mm <= 3) & (np.abs(rot_deg) < 30)
conds = {"left": rot_deg > 100, "right": rot_deg < -100}
print("Time points: slow walking", int(slow.sum()), "; left turns", int(conds["left"].sum()), "; right turns", int(conds["right"].sum()))
print("By fly:", {f: (int(slow[j * 3384:(j + 1) * 3384].sum()), int(conds["left"][j * 3384:(j + 1) * 3384].sum()),
                      int(conds["right"][j * 3384:(j + 1) * 3384].sum())) for j, f in enumerate(flies)})
col_of = {(p, b): c for c, (p, b) in enumerate(zip(block_plane, block_index))}
partner = np.array([col_of.get((p, int(bw.mirror_block(np.array([b]))[0])), -1) for p, b in zip(block_plane, block_index)])
print("Blocks with a kept mirror partner:", int((partner >= 0).sum()), "of", len(partner))
base = X[slow].mean(0)
turn_res = {}
for turn, m in conds.items():
    diff = X[m].mean(0) - base
    top = np.flatnonzero(diff >= np.percentile(diff, 90))
    top = top[partner[top] >= 0]
    pchange = diff[partner[top]]
    w = stats.wilcoxon(pchange)
    in_a = (block_bx[top] < nbx // 2).mean()
    turn_res[turn] = {"top blocks": len(top), "their change, median": float(np.median(diff[top])),
                      "mirror change, median": float(np.median(pchange)), "mirror change < 0 (%)": 100 * float((pchange < 0).mean()),
                      "Wilcoxon p": w.pvalue, "top blocks in half A (%)": 100 * in_a,
                      "all blocks: change range": (round(float(diff.min()), 3), round(float(diff.max()), 3))}
print(pd.DataFrame(turn_res).T.to_string())
opposite = (turn_res["left"]["top blocks in half A (%)"] > 50) != (turn_res["right"]["top blocks in half A (%)"] > 50)
ok = {t: r["mirror change, median"] < 0 and r["Wilcoxon p"] < 0.05 for t, r in turn_res.items()}
print("\nMirror decrease (median < 0, p < 0.05):", ok, "; increases in opposite halves:", opposite)
# Variant: other bounds for turning
for thr in (50, 150):
    vals = {}
    for turn, m in (("left", rot_deg > thr), ("right", rot_deg < -thr)):
        diff = X[m].mean(0) - base
        top = np.flatnonzero(diff >= np.percentile(diff, 90)); top = top[partner[top] >= 0]
        vals[turn] = (round(float(np.median(diff[partner[top]])), 4), float(f"{stats.wilcoxon(diff[partner[top]]).pvalue:.3g}"))
    print(f"Variant, turning threshold {thr} deg/s: mirror change median and p", vals)
# Per fly
rows = []
for j, f in enumerate(flies):
    sl = np.zeros(len(slow), bool); sl[j * 3384:(j + 1) * 3384] = True
    row = {"fly": f}
    for turn, m in conds.items():
        if (m & sl).sum() < 5 or (slow & sl).sum() < 5:
            row[turn] = np.nan
            continue
        diff = X[m & sl].mean(0) - X[slow & sl].mean(0)
        top = np.flatnonzero(diff >= np.percentile(diff, 90)); top = top[partner[top] >= 0]
        row[turn + " mirror median"] = float(np.median(diff[partner[top]]))
    rows.append(row)
print("\nPer fly (flies with at least 5 points in each condition):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced.** Turning raises activity mostly on one side, opposite for the two
directions (82% of the top 10% of blocks in half A for left turns, 25% for right turns), but
their mirror blocks rise too, by a median 0.0064 for left turns and 0.0022 for right turns, with
36% and 42% of them falling, against slow walking. The same holds with turning thresholds of 50
and 150 deg/s, and in 8 of 9 flies for left turns and 9 of 9 for right turns.

## Cross-correlation filters (Figure 6)

For each block, the behaviour at the neural time plus each shift (-5 to 5 s in 20 ms steps) is
weighted by the block signal and summed over time points and flies (linear_filters.py), notch
filtered at the volume rate, and deconvolved with the code's GCaMP6f kernel (deconvolution.py).
The lead is the shift at the peak of the deconvolved filter; positive means activity precedes
behaviour.

In [ ]:
def filters_for(offset_s=0.0):
    out = {key: np.zeros((len(bw.SHIFTS_S), X.shape[1])) for key in ("fwd", "left", "right")}
    for j, f in enumerate(flies):
        Xf = X[j * 3384:(j + 1) * 3384]
        for key in out:
            out[key] += bw.shifted_behaviour(by_fly[f], key, times[f] + offset_s) @ Xf
    return out

t0 = time.time()
raw_filters = filters_for(0.0)
rate = float(np.mean([s["processed"]["rate"] for s in sessions]))
b_notch, a_notch = signal.iirnotch(rate, Q=2, fs=1 / 0.02)
toe = bw.gcamp_toeplitz(len(bw.SHIFTS_S))
def peaks(raw):
    res = {}
    for key, flt in raw.items():
        notched = signal.filtfilt(b_notch, a_notch, flt, axis=0)
        dec = bw.deconvolve(notched.T, toe)                        # block, shift
        idx = dec.argmax(1)
        top = dec.max(1) >= np.percentile(dec.max(1), 60)          # top 40% by peak value
        res[key] = {"dec": dec, "lead": bw.SHIFTS_S[idx], "peak": dec.max(1), "top": top,
                    "lead_raw": bw.SHIFTS_S[notched.T.argmax(1)]}
    return res
pk = peaks(raw_filters)
print(f"Filters for {X.shape[1]} blocks in {time.time() - t0:.0f} s")

## fig6-peak-time-range

> The time of peak cross-correlation between supervoxel activity and forward velocity ranges from
> about 300 ms before the change in behaviour, the earliest, to more than 1 s after it.

In [ ]:
def lead_summary(res):
    rows = []
    for key, r in res.items():
        L = r["lead"][r["top"]]
        rows.append({"component": key, "blocks (top 40%)": int(r["top"].sum()),
                     "(a) earliest lead": L.max(), "(b) 99th pct lead": np.percentile(L, 99),
                     "median lead": np.median(L), "(b) 1st pct lead": np.percentile(L, 1), "(a) latest lead": L.min(),
                     "at window edge": int((np.abs(L) >= 4.9).sum()),
                     "median lead before deconvolution": np.median(r["lead_raw"][r["top"]])})
    return pd.DataFrame(rows)
leads = lead_summary(pk)
print("Lead in s (positive: activity precedes behaviour). Expected for forward velocity: earliest 0.3 s (0.27 to 0.33), latest a lag over 1 s")
print(leads.round(3).to_string(index=False))
fw = leads.set_index("component").loc["fwd"]
for rd, e, l in (("a", "(a) earliest lead", "(a) latest lead"), ("b", "(b) 99th pct lead", "(b) 1st pct lead")):
    print(f"Reading ({rd}): earliest within 0.27-0.33 s: {0.27 <= fw[e] <= 0.33}; latest lag > 1 s: {fw[l] < -1}")
hist = np.histogram(pk["fwd"]["lead"][pk["fwd"]["top"]], bins=np.arange(-5, 5.01, 0.25))
print("\nHistogram of VF leads (top 40%), 0.25 s bins, counts from -5 s:", hist[0].tolist())
# Variant: neural times at the start of each volume
pk_start = peaks(filters_for(-0.5 / rate))
print("\nVariant, neural times at the start of each volume:")
print(lead_summary(pk_start).round(3).to_string(index=False))
# Variant: top 20% and top 60%
for q in (80, 40):
    L = pk["fwd"]["lead"][pk["fwd"]["peak"] >= np.percentile(pk["fwd"]["peak"], q)]
    print(f"Variant, top {100 - q}% of VF filters: earliest {L.max():.2f}, 99th pct {np.percentile(L, 99):.2f}, "
          f"1st pct {np.percentile(L, 1):.2f}, latest {L.min():.2f} s")
# Per plane, VF
print("\nVF lead by plane (top 40%): median and range")
print(pd.DataFrame({"plane": block_plane[pk["fwd"]["top"]], "lead": pk["fwd"]["lead"][pk["fwd"]["top"]]})
      .groupby("plane").lead.agg(["size", "median", "min", "max"]).round(2).to_string())
print(f"Volume period: {1 / rate:.3f} s")
rows = []
for j, f in enumerate(flies):
    flt = bw.shifted_behaviour(by_fly[f], "fwd", times[f]) @ X[j * 3384:(j + 1) * 3384]
    dec = bw.deconvolve(signal.filtfilt(b_notch, a_notch, flt, axis=0).T, toe)
    t_ = dec.max(1) >= np.percentile(dec.max(1), 60)
    L = bw.SHIFTS_S[dec.argmax(1)][t_]
    rows.append({"fly": f, "99th pct lead": np.percentile(L, 99), "median lead": np.median(L), "1st pct lead": np.percentile(L, 1)})
print("\nPer fly, VF filters of that fly alone (top 40% within the fly):")
print(pd.DataFrame(rows).round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(pk["fwd"]["lead"][pk["fwd"]["top"]], bins=hist[1], color="0.4")
ax.set_xlabel("lead of activity over forward velocity (s)"); ax.set_ylabel("blocks")
plt.show()

**Result: reproduced.** Under reading (b), the 99th percentile of the VF lead is 0.32 s, within
0.27 to 0.33 s, and the 1st percentile is a lag of 1.23 s; under reading (a) the earliest lead is
0.44 s and the latest a lag of 1.44 s. For VL and VR the 99th percentiles are 0.30 and 0.22 s and
the 1st percentiles lags of 1.24 and 1.22 s. The value rests on the assumed neural times: with
the start of each volume as the neural time, the VF 99th percentile is 0.58 s. The median VF
lead also changes with FDA plane, from 0.36 s in plane 17 to -0.30 s in plane 80; the order in
which planes were acquired, which the dandiset does not give, can shift a plane by up to one
volume (0.532 s), so plane and timing cannot be separated here. Fitted per fly, the 99th
percentile is 0.36 to 4.55 s.

## fig6h-peak-vs-width

> Supervoxels whose peak correlation lags behaviour have broader temporal kernels, while those
> that anticipate behaviour have narrow kernels.

In [ ]:
top = pk["fwd"]["top"]
width = bw.fwhm(pk["fwd"]["dec"][top])
lag = -pk["fwd"]["lead"][top]
r_w = stats.pearsonr(lag, width)
print(f"VF, top 40% ({top.sum()} blocks): lag {lag.min():.2f} to {lag.max():.2f} s, FWHM {width.min():.2f} to {width.max():.2f} s")
print(f"Pearson r(lag, FWHM) = {r_w.statistic:.3f}, p = {r_w.pvalue:.3g}; Spearman {stats.spearmanr(lag, width).statistic:.3f}")
inner = np.abs(lag) < 4.5
print(f"Without filters peaking within 0.5 s of the window edge ({(~inner).sum()} dropped): r = {stats.pearsonr(lag[inner], width[inner]).statistic:.3f}")
for key in ("left", "right"):
    t_ = pk[key]["top"]
    print(f"{key}: r(lag, FWHM) = {stats.pearsonr(-pk[key]['lead'][t_], bw.fwhm(pk[key]['dec'][t_])).statistic:.3f}")
tb = pk_start["fwd"]["top"]
print(f"Variant, volume start times: r = {stats.pearsonr(-pk_start['fwd']['lead'][tb], bw.fwhm(pk_start['fwd']['dec'][tb])).statistic:.3f}")
print("Mean FWHM by lag bin (s):")
print(pd.Series(width).groupby(pd.cut(lag, [-5, -1, -0.5, 0, 0.5, 1, 5])).agg(["size", "mean"]).round(3).to_string())

**Result: reproduced.** Over the 2145 top VF blocks, lag and FWHM correlate at r = 0.672 (p =
2.2e-281; Spearman 0.703). Mean FWHM rises from 1.38 s for blocks leading behaviour by up to
0.5 s to 2.62 s for blocks lagging by more than 1 s. VL and VR give r = 0.442 and 0.449, and
volume-start times r = 0.673.

## fig6fg-ipsi-leads-contra

> Across nearly all brain regions, the peak correlation with angular velocity comes first in the
> hemisphere ipsilateral to the turn, then in the contralateral one.

In [ ]:
res_ic = {}
for key, comp in (("left", "left"), ("right", "right")):
    ipsi_is_a = ipsi_half[comp] == "A"
    t_ = pk[key]["top"]
    in_ipsi = (block_bx < nbx // 2) if ipsi_is_a else (block_bx >= nbx // 2)
    pairs = [(c, partner[c]) for c in np.flatnonzero(in_ipsi & t_) if partner[c] >= 0 and t_[partner[c]]]
    d = np.array([pk[key]["lead"][a] - pk[key]["lead"][b] for a, b in pairs])
    w = stats.wilcoxon(d) if len(d) else None
    res_ic[key] = {"ipsilateral half": ipsi_half[comp], "pairs": len(d), "median ipsi - contra lead (s)": float(np.median(d)),
                   "ipsi earlier (%)": 100 * float((d > 0).mean()), "ties (%)": 100 * float((d == 0).mean()),
                   "Wilcoxon p": w.pvalue}
print(pd.DataFrame(res_ic).T.to_string())
# By plane
for key in ("left", "right"):
    ipsi_is_a = ipsi_half[key] == "A"
    t_ = pk[key]["top"]
    in_ipsi = (block_bx < nbx // 2) if ipsi_is_a else (block_bx >= nbx // 2)
    rows = [(block_plane[c], pk[key]["lead"][c] - pk[key]["lead"][partner[c]])
            for c in np.flatnonzero(in_ipsi & t_) if partner[c] >= 0 and t_[partner[c]]]
    by = pd.DataFrame(rows, columns=["plane", "d"]).groupby("plane").d.agg(["size", "median"])
    print(f"\n{key}: median ipsi - contra lead by plane")
    print(by.round(3).T.to_string())

**Result: reproduced.** For VL, over 868 mirror pairs, the block in half A (the half with more VL
correlation) leads its partner by a median 0.16 s and is earlier in 67% of pairs; for VR, over
849 pairs, the block in half B leads by 0.14 s and is earlier in 72%; both Wilcoxon p-values
print as 0. The difference is largest in the planes from 17 to 38 (0.20 to 0.33 s) and near 0
in planes 73 and 80 (-0.03 to 0.03 s). Both blocks of a pair lie in the same FDA plane.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| dataset-structure | reproduced |
| fig2c-fraction-correlated | partially reproduced |
| fig2c-venn-categories | partially reproduced |
| fig2-symmetry | reproduced |
| fig2e-region-fractions | blocked: metadata insufficient |
| fig3-topographic-maps | blocked: metadata insufficient |
| fig4bcd-pc-structure | reproduced |
| fig4fg-ridge-r2 | partially reproduced |
| fig4h-r2-vs-movement | partially reproduced |
| fig4i-weights-stereotyped | reproduced |
| fig5a-unique-variance | partially reproduced |
| fig5c-global-below-threshold | reproduced |
| fig5f-ipsi-contra | not reproduced |
| fig6-peak-time-range | reproduced |
| fig6h-peak-vs-width | reproduced |
| fig6fg-ipsi-leads-contra | reproduced |
| fig6eg-region-sequence | blocked: metadata insufficient |
| fig7-within-region-gradients | blocked: metadata insufficient |
| figs2-tdtomato-control | blocked: data missing |
| figs4a-supervoxel-number | blocked: too large |

**How the analysis differs from the paper's.** It uses the authors' processed volumes in the FDA
space, not the paper's own mean brain, and 12 of the 91 FDA planes. The paper's supervoxels
(Ward clusters, 2000 per plane) are replaced by 6 x 6 voxel blocks. Behaviour is sampled at the
midpoint of each volume, as the dandiset has no plane times; the paper sampled it at each plane's
own time. Folds, the ridge penalty of the demixing models, the threshold for a unique
contribution, the bounds for slow walking and turning, and the notch filter's width are choices
made here, as neither the Methods nor the code fix them. The 1000 and 2000 PC models were not run.

**What was found about the dandiset.**

- It holds the raw and processed imaging and the ball tracking of all nine flies of the paper, and
  no recording twice. The processed volumes are in the Functional Drosophila Atlas, a space the
  preprint does not use, with voxel size given as 1, 1, 1 in an unknown unit.
- There are no neuropil labels for any of its spaces, and no mean brain, supervoxels or z-depth
  maps, so none of the paper's region-level results (Figures 2E to I, 3, 6E to G, 7) can be
  tested.
- Per-plane acquisition times are missing. The share of correlated voxels changes from 31% to 39%
  between the middle and the start of each volume, and the 99th percentile of forward-velocity lead
  from 0.32 to 0.58 s.
- Which half of the volume is the fly's left is not stated.
- The motion-corrected tdTomato channel, needed for the paper's motion control, is not included.
- Every FicTrac series has unit "meters" and conversion 0.0045, though most hold angles in
  radians; `SpatialSeriesRotationDeltaError`, described in radians, has medians of 6067 to 10,435.
- The paper's code computes the walking state with `np.sqrt(a, b)`, which uses only forward
  velocity; the line was written to combine forward and rotational velocity.